In [ ]:
# !pip install ninja ipykernel ipywidgets --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [1]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.12.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA H100 PCIe
VRAM: 79.2 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "meta-models/Muse-Glimmer-30B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.18.0.
To update, run: hf update
Fetching 13 files:   0%|                                | 0/13 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.2 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.2 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.2 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.2 seconds)
Fetching 13 files: 100%|███████████████████████| 13/13 [02:49<00:00, 13.08s/it]
Download complete: 100%|███████████████████| 59.6G/59.6G [02:49<00:00, 391MB/s]✓ Downloaded
  path: /workspace/local_model
Download complete: 100%|███████████████████| 59.6G/59.6G [02:50<00:00, 350MB/s]


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/1436 [00:00<?, ?it/s]

In [8]:
model

MuseGlimmerForConditionalGeneration(
  (model): MuseGlimmerModel(
    (vision_tower): MuseGlimmerVisionModel(
      (patch_embedder): MuseGlimmerVisionPatchEmbedder(
        (patch_embedding): Linear(in_features=1176, out_features=1536, bias=False)
        (position_embedding_table): Embedding(1024, 1536)
      )
      (rotary_emb): MuseGlimmerVisionRotaryEmbedding()
      (ln_pre): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
      (layers): ModuleList(
        (0-49): 50 x MuseGlimmerVisionEncoderLayer(
          (norm1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
          (attn): MuseGlimmerVisionAttention(
            (proj): Linear(in_features=1536, out_features=1536, bias=True)
            (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
            (k_proj): Linear(in_features=1536, out_features=1536, bias=True)
            (v_proj): L

In [ ]:
TUNING_CONFIG = {
    "group_size": 64,
    "sym": True,
    "iters": 800,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 2,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [ ]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:  # noqa: BLE001
        print(f"[Hub] ❌ Error uploading: {e}")

In [11]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-08-11 14:11:53 WARNING autoround.py L551: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-11 14:11:53 WARNING autoround.py L551: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-11 14:11:53 WARNING autoround.py L551: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [12]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq", inplace=True
)

2026-08-11 14:12:30 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.vision_tower.patch_embedder.patch_embedding
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-08-11 14:12:32 INFO orchestrator.py L570: start to cache block inputs
2026-08-11 14:12:32 INFO mllm.py L86: Using MLLM template: muse_glimmer
2026-08-11 14:12:32 INFO calib_dataset.py L1107: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-08-11 14:15:03 INFO device.py L1448: 'peak_ram': 72.16GB, 'peak_vram': 55.83GB
2026-08-11 14:15:03 INFO orchestrator.py L602: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/52 [00:11<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:882: UserWarning: cuDNN Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, war

Writing model shards:   0%|          | 0/6 [00:00<?, ?it/s]

packing: 100%|██████████| 417/417 [01:39<00:00,  4.19it/s]


Writing model shards:   0%|          | 0/6 [00:00<?, ?it/s]

2026-08-11 15:05:38 INFO device.py L1448: 'peak_ram': 74.26GB, 'peak_vram': 55.83GB


(MuseGlimmerForConditionalGeneration(
   (model): MuseGlimmerModel(
     (vision_tower): MuseGlimmerVisionModel(
       (patch_embedder): MuseGlimmerVisionPatchEmbedder(
         (patch_embedding): Linear(in_features=1176, out_features=1536, bias=False)
         (position_embedding_table): Embedding(1024, 1536)
       )
       (rotary_emb): MuseGlimmerVisionRotaryEmbedding()
       (ln_pre): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
       (layers): ModuleList(
         (0-49): 50 x MuseGlimmerVisionEncoderLayer(
           (norm1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
           (attn): MuseGlimmerVisionAttention(
             (proj): Linear(in_features=1536, out_features=1536, bias=True)
             (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
             (k_proj): Linear(in_features=1536, out_features=1536, bias=True)
      

In [13]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [16]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w4g64/auto-round-auto-gptq to Vishva007/Muse-Glimmer-30B-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Muse-Glimmer-30B-W4A16-AutoRound

[Hub] Pushing ./AutoRound/local_model-w4g64/auto-gptq to Vishva007/Muse-Glimmer-30B-W4A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Muse-Glimmer-30B-W4A16-AutoRound-GPTQ
